In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import Counter
import ast
from PIL import Image, ImageDraw
from wordcloud import WordCloud

# 设置绘图风格
chinese_fonts = ['Microsoft YaHei', 'SimHei', 'STHeiti', 'SimSun', 'Arial Unicode MS']
sns.set_theme(style="white", rc={"font.sans-serif": chinese_fonts})
plt.rcParams['font.sans-serif'] = chinese_fonts
plt.rcParams['axes.unicode_minus'] = False

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWS 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dws_community_daily 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_and_process_tags(city_name):
    """
    获取指定城市的标签数据并统计词频。
    """
    conn = get_db_connection()
    query = f"""
    SELECT hot_tags
    FROM iceberg_catalog.ershoufang.dws_community_daily
    WHERE city = '{city_name}' 
      AND hot_tags IS NOT NULL 
    """
    try:
        df = pd.read_sql(query, conn)
        
        # 统计词频
        tag_counter = Counter()
        for item in df['hot_tags']:
            if item is None: continue
            if isinstance(item, list):
                valid_tags = [str(t).strip() for t in item if str(t).strip()]
                tag_counter.update(valid_tags)
                
            elif isinstance(item, str):
                item = item.strip()
                # 过滤掉空的字符串数组
                if item in ('[]', '', 'null'):
                    continue
                try:
                    tags = ast.literal_eval(item)
                    if isinstance(tags, list):
                        valid_tags = [str(t).strip() for t in tags if str(t).strip()]
                        tag_counter.update(valid_tags)
                except:
                    continue
                
        return tag_counter
    finally:
        conn.close()

In [ ]:
def create_house_mask():
    """在内存中动态绘制一个房子形状的蒙版"""
    mask_img = Image.new('L', (800, 800), 255)
    draw = ImageDraw.Draw(mask_img)
    
    # 绘制房顶
    draw.polygon([(400, 100), (100, 400), (700, 400)], fill=0)
    # 绘制房体
    draw.rectangle([(200, 400), (600, 750)], fill=0)
    # 绘制烟囱
    draw.rectangle([(550, 150), (620, 300)], fill=0)
    
    return np.array(mask_img)

def plot_house_wordcloud(city):
    """绘制房子形状的词云图"""
    tag_counts = fetch_and_process_tags(city)
    
    if not tag_counts:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 暂无有效的标签数据", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    house_mask = create_house_mask()
    font_path = 'C:/Windows/Fonts/simhei.ttf' 
    
    try:
        wc = WordCloud(
            font_path=font_path,
            background_color='white',
            mask=house_mask,          
            max_words=200,            
            colormap='Spectral',      
            contour_width=2,          
            contour_color='steelblue',
            scale=3                   
        )
        
        wc.generate_from_frequencies(tag_counts)
        
    except OSError:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, "词云生成失败：未找到中文字体文件。\n请检查 font_path 路径是否正确配置。", 
                 ha='center', va='center', fontsize=12, color='red')
        plt.axis('off')
        plt.show()
        return

    # --- 开始绘图 ---
    fig, ax = plt.subplots(figsize=(10, 10))
    
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off') 
    
    plt.title(f'[{city}] 核心小区特色与优势高频词画像', fontsize=18, pad=20, fontweight='bold')
    plt.figtext(0.5, 0.05, "字号越大代表该标签在该城市小区中出现的频率越高", 
                ha='center', fontsize=11, color='gray')

    plt.tight_layout()
    plt.show()

In [19]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_house_wordcloud(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_house_wordcloud(default_city)

In [20]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()